        # 🔁 L06　迴圈 for / while
        **Python 冒險之旅 2026**　｜　Day 3（08/31 一）🌋 迴圈之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 5 章 5.1–5.4


        ### 🎯 這一關你會學到
        - 用 for 與 range() 重複執行
- 用 while 處理不確定次數的重複
- break、continue 與巢狀迴圈

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L06"
_SALT = "python-quest-2026-datama"
_TASKS = ["6-1", "6-2", "6-3", "6-4", "6-5", "6-6", "6-7"]
_XP_EACH = 14
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_6_1(run):
    out, ns = run()
    if "for" not in run.src: return (False, "要用 for 迴圈。")
    return (ns.get("total") == 5050, f"total 應該是 5050，現在是 {ns.get('total')}。")
任務定義("6-1", _check_6_1, 提示="range(1, 101) 才會包含 100。")

def _check_6_2(run):
    out, ns = run()
    lines = 行列表(out)
    if len(lines) < 2: return (False, "要有兩行輸出。")
    l1 = lines[0].split(); l2 = lines[1].split()
    if l1 != [str(i) for i in range(0, 21, 2)]: return (False, "第一行應該是 0 2 4 ... 20。")
    return (l2 == [str(i) for i in range(10, 0, -1)], "第二行應該是 10 9 8 ... 1。")
任務定義("6-2", _check_6_2, 提示="range(0, 21, 2) 與 range(10, 0, -1)。")

def _check_6_3(run):
    out, ns = run("abc", "xyz", "gotop")
    if out.count("帳號錯誤") != 2: return (False, "輸入 abc、xyz、gotop 時應該出現兩次 帳號錯誤!")
    return (出現(out, "帳號正確"), "最後要印出 帳號正確!")
任務定義("6-3", _check_6_3, 提示="if s == 'gotop': break，否則 print('帳號錯誤!')。")

def _check_6_4(run):
    if "continue" not in run.src: return (False, "要用 continue。")
    out, ns = run("a1b2c3")
    if "123" not in 行列表(out): return (False, "輸入 a1b2c3 應該印出 123。")
    out, ns = run("手機0912-345-678")
    return ("0912345678" in 行列表(out), "輸入 手機0912-345-678 應該印出 0912345678。")
任務定義("6-4", _check_6_4, 提示="判斷不是數字：not ch.isdigit()，或 ch > '9' or ch < '0'。")

def _check_6_5(run):
    out, ns = run()
    lines = 行列表(out)
    if len(lines) != 8: return (False, f"應該有 8 行（2～9），現在有 {len(lines)} 行。")
    o = _squash(out)
    for i in range(2, 10):
        for j in range(1, 10):
            if f"{i}*{j}={i*j}" not in o: return (False, f"少了 {i}*{j}={i*j}。")
    return True
任務定義("6-5", _check_6_5, 提示="外層 i 是 2～9，內層 j 是 1～9。")

def _check_6_6(run):
    out, ns = run()
    lines = [ln.rstrip() for ln in out.splitlines() if ln.strip()]
    want = [' ' * (5 - x) + '*' * x for x in range(1, 6)]
    return (lines == want, "每一行應該是 (5-x) 個空格加 x 顆星。")
任務定義("6-6", _check_6_6, 提示="print(' ' * (5 - x) + '*' * x)。")

def _check_6_7(run):
    out, ns = run("20")
    if 行列表(out)[-1].split() != ["2", "3", "5", "7", "11", "13", "17", "19"]: return (False, "輸入 20 應該印出 2 3 5 7 11 13 17 19。")
    out, ns = run("10")
    return (行列表(out)[-1].split() == ["2", "3", "5", "7"], "輸入 10 應該印出 2 3 5 7。")
任務定義("6-7", _check_6_7, 提示="如果 p % i == 0，表示 p 能被 i 整除，就不是質數。")


## 🔁 6-1　`for` 迴圈：知道要重複幾次就用它
```python
for 變數 in 序列:
    敘述      # 序列裡有幾個元素，就執行幾次
```
最常搭配 `range()`：
| 寫法 | 產生的數 |
|---|---|
| `range(5)` | 0, 1, 2, 3, 4（從 0 開始、不含 5） |
| `range(1, 6)` | 1, 2, 3, 4, 5 |
| `range(0, 11, 2)` | 0, 2, 4, 6, 8, 10（第三個是間隔） |
| `range(10, 0, -1)` | 10, 9, …, 1（倒數） |

In [ ]:
for x in 'Python':           # 課本 ex05/for_1.py：走訪字串
    print(x, end=' ')
print()
total = 0
for x in range(1, 11):       # 課本 ex05/for_2.py：1 加到 10
    total += x
print('1~10 的總和 =', total)
for i in range(10, 0, -2):
    print(i, end=' ')
print()
for x in 'abc':              # for…else：迴圈正常跑完才執行 else
    print(x)
else:
    print('字串輸出完畢!')

## 6-2　`while` 迴圈：不知道要幾次，條件成立就一直做
```python
while 條件:
    敘述       # 每次做完都回頭檢查條件，False 才停
```
⚠️ 記得在迴圈裡**改變條件**，否則會變成無窮迴圈（Colab 左邊一直轉圈 → 按「中斷執行」）。
`while True:` 搭配 `break` 是常見的「直到使用者說停為止」寫法。

In [ ]:
i, total = 1, 0
while i <= 10:               # 課本 ex05/while_1.py
    total += i
    i += 1
print(total)
i = 3
while i > 0:                 # while…else
    print(i)
    i -= 1
else:
    print('時間到！')

## 6-3　`break` 與 `continue`
- `break`：**立刻跳出**整個迴圈。
- `continue`：跳過本次剩下的敘述，**直接進入下一次**。

In [ ]:
while True:                              # 課本 ex05/break.py
    s = input('請輸入帳號：')
    if s == 'gotop':
        break
    print('帳號錯誤!')
print('帳號正確!')

In [ ]:
s = input("請輸入字串：")                  # 課本 ex05/continue.py：只留下數字
for ch in s:
    if ch > '9' or ch < '0':
        continue
    print(ch, end='')
print()

## 6-4　巢狀迴圈：迴圈裡面還有迴圈
外層每跑一次，內層就完整跑一輪。九九乘法表、畫圖形、處理表格都靠它。

In [ ]:
for x in range(1, 6):                    # 課本 ex05/n_loop.py：星星三角形
    for y in range(1, 6 - x):
        print(' ', end='')
    for y in range(1, x + 1):
        print('*', end='')
    print()
for i in range(1, 4):
    for j in range(1, 4):
        print(f'{i}*{j}={i*j}', end='\t')
    print()

### 🎯 任務 6-1　加總機

用 `for` 和 `range()` 把 1 到 100 加總，印出 `1~100 的總和 = 5050`。

In [ ]:
# 🎯 任務 6-1　加總機（請保留這一行）
total = 0
for i in range(???):
    total += i
print("1~100 的總和 =", total)

In [ ]:
檢查("6-1")   # ◀ 執行這一格，看看任務 6-1 有沒有過關

### 🎯 任務 6-2　偶數與倒數

印出兩行：第一行是 0 到 20 的所有**偶數**（空格隔開），第二行從 10 **倒數**到 1（空格隔開）。

In [ ]:
# 🎯 任務 6-2　偶數與倒數（請保留這一行）
for i in range(???):
    print(i, end=' ')
print()
for i in range(???):
    print(i, end=' ')
print()

In [ ]:
檢查("6-2")   # ◀ 執行這一格，看看任務 6-2 有沒有過關

### 🎯 任務 6-3　帳號守門員

用 `while True` 不斷讀取帳號，直到輸入 `gotop` 為止；輸入錯誤時印 `帳號錯誤!`，正確後跳出迴圈並印 `帳號正確!`。

In [ ]:
# 🎯 任務 6-3　帳號守門員（請保留這一行）
while True:
    s = input('請輸入帳號：')
    # 正確就 break，否則印出錯誤訊息
print('帳號正確!')

In [ ]:
檢查("6-3")   # ◀ 執行這一格，看看任務 6-3 有沒有過關

### 🎯 任務 6-4　數字過濾器

讀取一個字串，用 `continue` 跳過不是數字的字元，只把數字字元印在同一行（例如輸入 `a1b2c3` → `123`）。

In [ ]:
# 🎯 任務 6-4　數字過濾器（請保留這一行）
s = input("請輸入字串：")
for ch in s:
    if ???:
        continue
    print(ch, end='')
print()

In [ ]:
檢查("6-4")   # ◀ 執行這一格，看看任務 6-4 有沒有過關

### 🎯 任務 6-5　九九乘法表

用巢狀迴圈印出 2 到 9 的乘法表：每一行是同一個乘數（例如 `2*1=2  2*2=4 ... 2*9=18`），一共 8 行。格式用 `f'{i}*{j}={i*j}'`，用 `\t` 或空格隔開。

In [ ]:
# 🎯 任務 6-5　九九乘法表（請保留這一行）
for i in range(2, 10):
    for j in range(???):
        print(f"{i}*{j}={i*j}", end="\t")
    print()

In [ ]:
檢查("6-5")   # ◀ 執行這一格，看看任務 6-5 有沒有過關

### 🎯 任務 6-6　金字塔

印出 5 層靠右對齊的星星三角形（第 1 層 1 顆、第 5 層 5 顆，前面補空格）：
```
    *
   **
  ***
 ****
*****
```
提示：可以用兩個內層迴圈，也可以用字串乘法 `' ' * n + '*' * m`。

In [ ]:
# 🎯 任務 6-6　金字塔（請保留這一行）
for x in range(1, 6):
    # 印出 5-x 個空格，再印 x 顆星，最後換行

In [ ]:
檢查("6-6")   # ◀ 執行這一格，看看任務 6-6 有沒有過關

### 🎯 任務 6-7　質數列表

讀取一個正整數 n，印出 2 到 n 之間所有的質數（同一行，用空格隔開）。質數：只能被 1 和自己整除。輸入 20 → `2 3 5 7 11 13 17 19`

In [ ]:
# 🎯 任務 6-7　質數列表（請保留這一行）
n = int(input('請輸入一個正整數：'))
for p in range(2, n + 1):
    is_prime = True
    for i in range(2, p):
        if ???:
            is_prime = False
            break
    if is_prime:
        print(p, end=' ')
print()

In [ ]:
檢查("6-7")   # ◀ 執行這一格，看看任務 6-7 有沒有過關

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：📋 L07 串列 list** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L07_lists.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/